In [1]:
from config import data
from groq_client import call_groq
import json 
import os

## Ticket Summarization

In [2]:
from prompts import system_ticket_summary, user_ticket_summary_template

In [3]:
ticket_summary_prompt = [
    {"role" : "system", "content" : system_ticket_summary},
    {"role" : "user", "content" : user_ticket_summary_template.format(data=data)}
]

ticket_summary = call_groq(prompt=ticket_summary_prompt)

json.loads(ticket_summary)

{'short_summary': 'Customer was charged twice after cancelling a premium subscription and has not received a response from support.',
 'customer_problem': 'A double charge appears on the bank statement for a subscription that was cancelled last month, and previous support inquiries were unanswered.',
 'business_impact': "Financial loss for the customer, risk of public escalation on LinkedIn, and potential blockage of future payments by the customer's finance team.",
 'customer_requested_action': 'Refund the incorrect charge immediately and resolve the issue today.',
 'important_context': ['Premium subscription cancelled last month.',
  'Charge and duplicate invoice appeared this month.',
  'Customer contacted support twice last week with no proper response.',
  'Ticket is for Billing and subscription.',
  'Customer is a Premium tier client.'],
 'missing_information': ['Invoice ID or reference number for the duplicate charge.',
  'Registered account email or customer ID to verify cancel

## Ticket Classification 

In [4]:
from prompts import system_ticket_classification, user_ticket_classification_template

In [5]:
ticket_classification_prompt = [
    {"role" : "system", "content" : system_ticket_classification},
    {"role" : "user", "content" : user_ticket_classification_template.format(data=data)}
]

ticket_classification = call_groq(prompt=ticket_classification_prompt)

json.loads(ticket_classification)

{'primary_category': 'BILLING_ISSUE',
 'secondary_categories': ['CANCELLATION_OR_REFUND', 'ESCALATION_COMPLAINT'],
 'category_reasoning_summary': 'The customer reports being charged twice after cancelling a premium subscription and requests an immediate refund, indicating a clear billing problem. They also mention lack of response and threat to escalate, adding a cancellation/refund aspect and an escalation complaint.',
 'confidence_score': 0.97}

## Ticket Sentiment

In [6]:
from prompts import user_ticket_sentiment_template, system_ticket_sentiment

In [7]:
ticket_sentiment_prompt = [
    {"role" : "system", "content" : system_ticket_sentiment},
    {"role" : "user", "content" : user_ticket_sentiment_template.format(data=data)}
]

ticket_sentiment = call_groq(prompt=ticket_sentiment_prompt)

json.loads(ticket_sentiment)

{'sentiment': 'URGENT',
 'emotion_signals': ['repeated follow-ups',
  'threat of escalation',
  'urgency',
  'frustration',
  'dissatisfaction',
  'loss of trust'],
 'sentiment_reasoning_summary': 'The customer reports being charged twice after cancellation, mentions two prior support contacts with no proper response, and threatens public escalation and finance block if not resolved today. This combination of repeated follow‑ups, escalation threat, and a demand for immediate action signals a high‑urgency, frustrated sentiment.',
 'confidence_score': 0.96}

## Ticket Priority & Escalation Risk

In [8]:
from prompts import system_ticket_priority_escalation, user_ticket_priority_escalation_template

In [9]:
ticket_priority_escalation_prompt = [
    {"role" : "system", "content" : system_ticket_priority_escalation},
    {"role" : "user", "content" : user_ticket_priority_escalation_template.format(
        data=data,
        sentiment=ticket_sentiment
    )}
]

ticket_priority_escalation = call_groq(prompt=ticket_priority_escalation_prompt)

json.loads(ticket_priority_escalation)

{'priority': 'URGENT',
 'escalation_risk': 'CRITICAL',
 'risk_triggers': ['double charge after cancellation',
  'payment/financial impact',
  'multiple unanswered support attempts',
  'threat of public escalation on LinkedIn',
  'threat to block future payments by finance team',
  'premium customer status'],
 'recommended_sla_action': 'Escalate immediately to Billing Support Lead and Account Management; acknowledge the ticket within 30 minutes, begin verification of the duplicate charge, and initiate refund process. Provide hourly status updates to the customer and involve a senior manager to mitigate public escalation risk.',
 'reasoning_summary': 'The premium customer faces a direct financial impact (duplicate charge) and has experienced repeated support failures. They have explicitly threatened a public escalation and finance block, raising both business and reputational risk. Given the SLA tier, payment impact, repeated follow‑ups, and escalation threat, the ticket warrants an URGE

## Sensitive Information

In [10]:
from prompts import system_ticket_sensitive_information, user_ticket_sensitive_information_template

In [11]:
ticket_sensitive_information_prompt = [
    {"role" : "system", "content" : system_ticket_sensitive_information},
    {"role" : "user", "content" : user_ticket_sensitive_information_template.format(data=data)}
]

ticket_sensitive_information = call_groq(prompt=ticket_sensitive_information_prompt)

json.loads(ticket_sensitive_information)

{'sensitive_information_detected': True,
 'sensitive_categories': ['PERSONAL_INFORMATION'],
 'evidence_summary': "The ticket contains the customer's name (Amit) and references to billing details (duplicate charge, invoice) that could be linked to the individual's account.",
 'handling_recommendations': ["Verify the customer's identity before proceeding with any refund or account changes.",
  'Request the specific invoice ID or registered email associated with the account to locate the transaction.',
  "Do not disclose the customer's personal name or billing details in any public or insecure channels.",
  'Escalate the issue to the billing support team as per the business rules for premium customers.',
  'Follow the rule: Do not promise a refund before verification and do not confirm cancellation until verified.']}

## Internal Routing

In [12]:
from prompts import system_ticket_internal_routing, user_ticket_internal_routing_template

In [13]:
ticket_internal_routing_prompt = [
    {"role" : "system", "content" : system_ticket_internal_routing},
    {"role" : "user", "content" : user_ticket_internal_routing_template.format(data=data)},
]

ticket_internal_routing = call_groq(prompt=ticket_internal_routing_prompt)

json.loads(ticket_internal_routing)

{'recommended_team': 'BILLING_SUPPORT',
 'routing_reason': 'Premium customer reporting duplicate charge and unresponsive prior support; requires billing verification, potential refund, and escalation per business rules.',
 'internal_note': "Verify the customer's account email and the specific invoice ID(s) showing the duplicate charge. Confirm the cancellation status of the premium subscription before any refund is processed. Follow the business rule: do not promise a refund until verification is complete. Respond professionally and empathetically, and keep the customer updated on the investigation.",
 'required_follow_up_information': ['Invoice ID(s) showing the duplicate charge',
  'Registered account email address',
  'Last four digits of the payment method used',
  'Confirmation of the cancellation request date']}

## Draft Customer Response

In [14]:
from prompts import system_customer_response, user_customer_response_template

In [15]:
customer_response_prompt = [
    {"role" : "system", "content" : system_customer_response},
    {"role" : "user", "content" : user_customer_response_template.format(data=data)}
]

customer_response = call_groq(prompt=customer_response_prompt)

json.loads(customer_response)

{'draft_response': 'Hi Amit,\n\nThank you for reaching out and I’m sorry to hear about the duplicate charge and the delay in response – I understand how frustrating this must be.\n\nI see that you cancelled your Premium subscription last month and that the same invoice amount appears twice on your bank statement. To investigate this promptly, could you please provide the invoice number(s) and the email address associated with your account (or the subscription ID)? This information will allow us to verify the cancellation status and the charges.\n\nOnce we have those details, I will forward your case to our Billing Support team, who will prioritize a review and get back to you with a resolution as quickly as possible.\n\nWe appreciate your patience and will keep you updated.\n\nBest regards,\n[Your Name]\nCustomer Support',
 'response_strategy': "1. Acknowledge the customer's concern and frustration. 2. Express empathy for the duplicate charge and lack of response. 3. Summarize the issu

## Response Quality Review

In [16]:
from prompts import system_quality_review, user_quality_review_template

In [17]:
quality_review_prompt = [
    {"role" : "system", "content" : system_quality_review},
    {"role" : "user", "content" : user_quality_review_template.format(response=customer_response)}
]

quality_review = call_groq(prompt=quality_review_prompt)

json.loads(quality_review)

{'scores': {'empathy': 5,
  'correctness': 5,
  'actionability': 5,
  'policy_safety': 5,
  'tone_alignment': 5,
  'completeness': 5},
 'strengths': ["Strong empathy statement that acknowledges the customer's frustration.",
  'Clear, actionable request for specific information (invoice numbers, email/subscription ID).',
  'Safe language that avoids making unsupported promises or guarantees.',
  'Professional and courteous tone throughout the response.'],
 'improvement_areas': ['Could include an estimated timeline for the follow‑up (e.g., "we aim to get back to you within 2 business days").',
  'Might reference any prior tickets or communications to show continuity, if applicable.'],
 'final_review_summary': "The draft response effectively acknowledges Amit's frustration, stays fact‑based, provides clear next steps, and adheres to policy safety while maintaining a professional tone. Minor enhancements could involve adding an expected response timeframe and referencing any earlier intera

## Final Ticket

In [18]:
from output_parser import parse_json_if_needed

In [24]:
final_ticket = {
    "ticket_summary": parse_json_if_needed(ticket_summary),
    "classification": parse_json_if_needed(ticket_classification),
    "sentiment_analysis": parse_json_if_needed(ticket_sentiment),
    "priority_and_risk": parse_json_if_needed(ticket_priority_escalation),
    "sensitive_information_check": parse_json_if_needed(ticket_sensitive_information),
    "routing_recommendation": parse_json_if_needed(ticket_internal_routing),
    "draft_customer_response": parse_json_if_needed(customer_response),
    "response_quality_review": parse_json_if_needed(quality_review),
    "generation_metadata": {
        "model_used": "llama-3.3-70b-versatile",
        "temperature": 0.2,
        "total_steps_completed": 8
    }
}

print(final_ticket)

try:
    os.makedirs("output", exist_ok=True)

    with open("output/final_ticket.json", "w", encoding="utf-8") as f:
        json.dump(final_ticket, f, indent=4, ensure_ascii=False)

    print("saved successfully.")
except Exception as e:
    print(e)

{'ticket_summary': {'short_summary': 'Customer was charged twice after cancelling a premium subscription and has not received a response from support.', 'customer_problem': 'A double charge appears on the bank statement for a subscription that was cancelled last month, and previous support inquiries were unanswered.', 'business_impact': "Financial loss for the customer, risk of public escalation on LinkedIn, and potential blockage of future payments by the customer's finance team.", 'customer_requested_action': 'Refund the incorrect charge immediately and resolve the issue today.', 'important_context': ['Premium subscription cancelled last month.', 'Charge and duplicate invoice appeared this month.', 'Customer contacted support twice last week with no proper response.', 'Ticket is for Billing and subscription.', 'Customer is a Premium tier client.'], 'missing_information': ['Invoice ID or reference number for the duplicate charge.', 'Registered account email or customer ID to verify ca